# Install Dependencies

In [ ]:
!pip install openai

In [ ]:
!pip install transformers torch scikit-learn matplotlib seaborn bert-score nltk rouge_score

# Import Dependencies

In [ ]:
import openai
from openai import OpenAI
from google.colab import userdata
from IPython.display import Markdown as md
import pandas as pd
import json

In [ ]:
client = OpenAI(
  api_key=userdata.get('OPENAI_API_KEY')
)

# ChatFolio System Prompt

This is the main prompt from `chat_engine.py` that powers the conversational profile extraction. This is the prompt under test.

In [ ]:
system_prompt = '''\
You are ChatFolio, a friendly investment advisor that helps beginners build a starter portfolio through conversation.

Your job is to learn about the user and fill their investment profile. The profile has these fields:
- goal: What they're investing for (retirement, house, emergency fund, education, wealth building, etc.)
- timeline: How many years until they need the money
- monthly_budget: How much they can invest per month (USD)
- risk_tolerance: conservative, moderate, or aggressive
- additional_info: Any extra preferences or constraints (e.g., "no crypto", "already have a 401k", "prefer ESG funds")

Guidelines:
1. Introduce yourself briefly and ask about their investment goal. Keep it warm but short.
2. Ask ONE question at a time. Never stack multiple questions.
3. When a user's answer is vague or could mean different things, paraphrase your understanding and ask them to confirm before saving it.
4. Only include a field in profile_updates when you're confident the user has confirmed or clearly stated it.
5. After the 4 main fields are filled, ask: "Is there anything else you'd like me to know?"
6. If the user says they have nothing to add, set additional_info to "None specified" and set ready_for_portfolio to true.
7. If the user DOES share extra info, save it to additional_info and then set ready_for_portfolio to true.
8. When ready_for_portfolio is true, tell the user they can click the button below to generate their portfolio.
9. Use plain language. If you use a financial term, explain it simply.
10. Keep responses to 2-3 sentences unless the user asks for an explanation.
11. Be encouraging. Investing is intimidating for beginners - make it feel approachable.
12. You are an educational tool, NOT a licensed financial advisor.
13. AFTER the portfolio has been generated, the user may continue chatting to request changes.

You MUST respond with valid JSON in this exact format:
{
  "message": "Your conversational reply",
  "profile_updates": {
    "goal": "value or null",
    "timeline": "value or null",
    "monthly_budget": "value or null",
    "risk_tolerance": "value or null",
    "additional_info": "value or null"
  },
  "ready_for_portfolio": false,
  "profile_changed": false
}

Rules for profile_updates:
- Use null for fields you are NOT updating this turn.
- Only set a value when the user has clearly stated or confirmed it.
- For timeline, store as a readable string like "30 years" or "5-10 years".
- For monthly_budget, store just the number as a string like "150" or "500".
- For risk_tolerance, only use: "conservative", "moderate", or "aggressive".
'''

# Conversation Stage Contexts

Each test question is asked at a specific stage. To simulate realistic conversations, we set up the appropriate profile state and conversation history for each stage.

In [ ]:
stage_contexts = {
    "Goal": {
        "profile": {
            "goal": None, "timeline": None,
            "monthly_budget": None, "risk_tolerance": None,
            "additional_info": None
        },
        "history": [
            {"role": "assistant", "content": "Hey! I'm ChatFolio. I help you build a starter investment portfolio through conversation. What are you looking to invest for?"}
        ]
    },
    "Timeline": {
        "profile": {
            "goal": "Retirement", "timeline": None,
            "monthly_budget": None, "risk_tolerance": None,
            "additional_info": None
        },
        "history": [
            {"role": "assistant", "content": "Hey! I'm ChatFolio. I help you build a starter investment portfolio through conversation. What are you looking to invest for?"},
            {"role": "user", "content": "I want to save for retirement."},
            {"role": "assistant", "content": "Retirement is a great goal! How many years until you plan to retire?"}
        ]
    },
    "Budget": {
        "profile": {
            "goal": "Retirement", "timeline": "30 years",
            "monthly_budget": None, "risk_tolerance": None,
            "additional_info": None
        },
        "history": [
            {"role": "assistant", "content": "Hey! I'm ChatFolio. What are you looking to invest for?"},
            {"role": "user", "content": "I want to save for retirement."},
            {"role": "assistant", "content": "Retirement is a great goal! How many years until you plan to retire?"},
            {"role": "user", "content": "About 30 years."},
            {"role": "assistant", "content": "30 years is a great timeline! How much can you invest per month?"}
        ]
    },
    "Risk": {
        "profile": {
            "goal": "Retirement", "timeline": "30 years",
            "monthly_budget": "200", "risk_tolerance": None,
            "additional_info": None
        },
        "history": [
            {"role": "assistant", "content": "Hey! I'm ChatFolio. What are you looking to invest for?"},
            {"role": "user", "content": "I want to save for retirement."},
            {"role": "assistant", "content": "How many years until you plan to retire?"},
            {"role": "user", "content": "About 30 years."},
            {"role": "assistant", "content": "How much can you invest per month?"},
            {"role": "user", "content": "$200 a month."},
            {"role": "assistant", "content": "$200/month is a solid start! How do you feel about risk — would you say conservative, moderate, or aggressive?"}
        ]
    },
    "Additional": {
        "profile": {
            "goal": "Retirement", "timeline": "30 years",
            "monthly_budget": "200", "risk_tolerance": "moderate",
            "additional_info": None
        },
        "history": [
            {"role": "assistant", "content": "Hey! I'm ChatFolio. What are you looking to invest for?"},
            {"role": "user", "content": "I want to save for retirement."},
            {"role": "assistant", "content": "How many years until you plan to retire?"},
            {"role": "user", "content": "About 30 years."},
            {"role": "assistant", "content": "How much can you invest per month?"},
            {"role": "user", "content": "$200 a month."},
            {"role": "assistant", "content": "How do you feel about risk?"},
            {"role": "user", "content": "I'd say moderate."},
            {"role": "assistant", "content": "Moderate is a great balanced choice! Is there anything else you'd like me to know? For example, any investment preferences, accounts you already have, or things you want to avoid?"}
        ]
    },
    "Post-Generation": {
        "profile": {
            "goal": "Retirement", "timeline": "30 years",
            "monthly_budget": "200", "risk_tolerance": "moderate",
            "additional_info": "None specified"
        },
        "history": [
            {"role": "assistant", "content": "Hey! I'm ChatFolio. What are you looking to invest for?"},
            {"role": "user", "content": "I want to save for retirement."},
            {"role": "assistant", "content": "How many years until you plan to retire?"},
            {"role": "user", "content": "About 30 years."},
            {"role": "assistant", "content": "How much can you invest per month?"},
            {"role": "user", "content": "$200 a month."},
            {"role": "assistant", "content": "How do you feel about risk?"},
            {"role": "user", "content": "I'd say moderate."},
            {"role": "assistant", "content": "Is there anything else you'd like me to know?"},
            {"role": "user", "content": "No, that's everything."},
            {"role": "assistant", "content": "You're all set! Click the Generate My Portfolio button below to see your personalized recommendation."}
        ]
    },
    "N/A": {
        "profile": {
            "goal": None, "timeline": None,
            "monthly_budget": None, "risk_tolerance": None,
            "additional_info": None
        },
        "history": [
            {"role": "assistant", "content": "Hey! I'm ChatFolio. I help you build a starter investment portfolio through conversation. What are you looking to invest for?"}
        ]
    }
}

# Helper: Call ChatFolio LLM

This function sends a user message to GPT-4o-mini with the ChatFolio system prompt and the appropriate conversation context for the given stage.

In [ ]:
def get_chatfolio_response(question, stage):
    """Send a test question to ChatFolio and return the parsed JSON response."""
    ctx = stage_contexts[stage]
    profile_context = (
        "\n\nCurrent user profile:\n"
        + json.dumps(ctx["profile"], indent=2)
        + "\nOnly update fields the user has clearly confirmed."
    )

    messages = [
        {"role": "system", "content": system_prompt + profile_context}
    ] + ctx["history"] + [
        {"role": "user", "content": question}
    ]

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        response_format={"type": "json_object"},
        temperature=0.7,
        max_tokens=500,
    )

    try:
        parsed = json.loads(response.choices[0].message.content)
        return parsed.get("message", response.choices[0].message.content)
    except json.JSONDecodeError:
        return response.choices[0].message.content

# Load Benchmark Dataset

In [ ]:
df = pd.read_csv('benchmark_dataset.csv')
print(f"Total samples: {len(df)}")
df.head()

# Evaluate Relevancy Classification

We use the LLM to classify whether each question is relevant to investment profile building or off-topic.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class ClassificationResult(BaseModel):
    relevancy: Literal['Profile Building', 'Off-Topic'] = Field(
        description='Is the question relevant to building an investment portfolio?'
    )

for index, row in df.iterrows():
    question = row['Question']

    classification_prompt = f'''
    You are ChatFolio, an AI investment advisor that helps users build a starter portfolio.
    Your scope is limited to: investment goals, timelines, budgets, risk tolerance, and portfolio questions.

    Determine if the following user message is relevant to building an investment portfolio ("Profile Building")
    or completely unrelated ("Off-Topic").

    User message: {question}
    '''

    response = client.responses.parse(
        model='gpt-4o-mini',
        input=classification_prompt,
        text_format=ClassificationResult
    )

    df.at[index, 'Predicted Relevancy'] = response.output_parsed.relevancy

df[['Question', 'Relevancy', 'Predicted Relevancy']].head(10)

## Confusion Matrix & Classification Metrics

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt

precision = precision_score(df['Relevancy'], df['Predicted Relevancy'], pos_label='Profile Building')
recall = recall_score(df['Relevancy'], df['Predicted Relevancy'], pos_label='Profile Building')
f1 = f1_score(df['Relevancy'], df['Predicted Relevancy'], pos_label='Profile Building')
accuracy = accuracy_score(df['Relevancy'], df['Predicted Relevancy'])

conf_matrix = confusion_matrix(df['Relevancy'], df['Predicted Relevancy'],
                                labels=['Profile Building', 'Off-Topic'])

plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Profile Building', 'Off-Topic'],
            yticklabels=['Profile Building', 'Off-Topic'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Relevancy Classification\nAccuracy: {accuracy:.2f}  Precision: {precision:.2f}  Recall: {recall:.2f}  F1: {f1:.2f}')
plt.show()

# Evaluate Response Quality

## Step 1: Generate LLM Responses (V1)

Run each test question through ChatFolio's system prompt with the appropriate conversation context for its stage.

In [ ]:
for index, row in df.iterrows():
    question = row['Question']
    stage = row['Stage']

    response = get_chatfolio_response(question, stage)
    df.at[index, 'LLM Response V1'] = response

    if index % 10 == 0:
        print(f"Processed {index + 1}/{len(df)}...")

print("Done!")
df[['Question', 'Stage', 'Ground Truth Response', 'LLM Response V1']].head()

## Step 2: LLM-as-Judge Evaluation (V1)

Use a separate LLM call to rate each response against the ground truth on a 1-5 scale.

In [ ]:
class Rating(BaseModel):
    rating: Literal[1, 2, 3, 4, 5] = Field(
        description='Rate the quality of the LLM response compared to the ground truth on a scale of 1-5.'
    )

for index, row in df.iterrows():
    llm_response = row['LLM Response V1']
    ground_truth = row['Ground Truth Response']

    eval_prompt = f'''
    Rate the quality of the LLM response compared to the ground truth reference on a scale of 1-5.

    5 - Excellent: The response fully matches the expected behavior described in the ground truth.
    4 - Good: The response is mostly correct with minor differences from the ground truth.
    3 - Adequate: The response is partially correct but misses key elements of the ground truth.
    2 - Poor: The response mostly fails to match the expected behavior.
    1 - Unsatisfactory: The response is irrelevant or contradicts the ground truth.

    Ground Truth Reference: {ground_truth}
    LLM Response: {llm_response}
    '''

    response = client.responses.parse(
        model='gpt-4o-mini',
        input=eval_prompt,
        text_format=Rating
    )

    df.at[index, 'LLM V1 Rating'] = response.output_parsed.rating

print(f"Average LLM V1 Rating: {df['LLM V1 Rating'].mean():.2f}")
df[['Question', 'Ground Truth Response', 'LLM Response V1', 'LLM V1 Rating']].head()

## Step 3: Manual Human Evaluation

Add your own 1-5 ratings to compare with the LLM judge. You can:
1. Use the interactive Colab sheet
2. Export the CSV, rate in a spreadsheet, and re-import
3. Enter ratings directly in code

In [ ]:
# Option 1: Interactive sheet (uncomment in Colab)
# from google.colab import sheets
# sheet = sheets.InteractiveSheet(df=df)

# Option 2: Export, edit externally, re-import
# df.to_csv('benchmark_dataset_with_ratings.csv', index=False)
# df = pd.read_csv('benchmark_dataset_with_ratings.csv')

# Option 3: Enter ratings manually (replace with your actual ratings)
# df['Human V1 Rating'] = [your_ratings_here]

df.head()

## Export Results

In [ ]:
df.to_csv('benchmark_dataset_with_ratings.csv', index=False)
print("Exported to benchmark_dataset_with_ratings.csv")
df.head()

## Step 4: Compute Alignment Between Human and LLM Ratings

These correlation scores measure whether the LLM judge agrees with human judgment.

In [ ]:
from scipy.stats import kendalltau, spearmanr

# Only run this after filling in Human V1 Rating
if 'Human V1 Rating' in df.columns and df['Human V1 Rating'].notna().all():
    tau, tau_p = kendalltau(df['LLM V1 Rating'], df['Human V1 Rating'])
    rho, rho_p = spearmanr(list(df['LLM V1 Rating']), list(df['Human V1 Rating']))

    print(f"Kendall's Tau:  {tau:.3f}  (p={tau_p:.4f})")
    print(f"Spearman's Rho: {rho:.3f}  (p={rho_p:.4f})")
else:
    print("Fill in 'Human V1 Rating' column first, then re-run this cell.")

# Additional Metrics

These metrics compare LLM responses against ground truth using different similarity measures.
Note: You cannot compare scores across different metrics — only use them to compare V1 vs V2 within the same metric.

## BERTScore

In [ ]:
from bert_score import score

P, R, F1 = score(
    list(df['LLM Response V1']),
    list(df['Ground Truth Response']),
    lang="en", verbose=True
)

print(f"BERTScore — Precision: {P.mean():.3f}  Recall: {R.mean():.3f}  F1: {F1.mean():.3f}")

## BLEU

In [ ]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import nltk
nltk.download('punkt_tab', quiet=True)

def compute_bleu(reference, candidate):
    smooth = SmoothingFunction().method1
    return sentence_bleu([reference.split()], candidate.split(), smoothing_function=smooth)

bleu_scores = df.apply(
    lambda row: compute_bleu(row['Ground Truth Response'], row['LLM Response V1']),
    axis=1
)

print(f"Average BLEU Score: {bleu_scores.mean():.4f}")

## ROUGE

In [ ]:
from rouge_score import rouge_scorer

def compute_rouge(reference, candidate):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    return scorer.score(reference, candidate)

rouge_scores = df.apply(
    lambda row: compute_rouge(row['Ground Truth Response'], row['LLM Response V1']),
    axis=1
)

rouge1 = [s['rouge1'].recall for s in rouge_scores]
rouge2 = [s['rouge2'].recall for s in rouge_scores]
rougeL = [s['rougeL'].recall for s in rouge_scores]

print(f"Average ROUGE-1 Recall: {sum(rouge1)/len(rouge1):.4f}")
print(f"Average ROUGE-2 Recall: {sum(rouge2)/len(rouge2):.4f}")
print(f"Average ROUGE-L Recall: {sum(rougeL)/len(rougeL):.4f}")

# Summary

| Metric | V1 Score |
|--------|----------|
| LLM Judge (avg 1-5) | *fill after running* |
| BERTScore F1 | *fill after running* |
| BLEU | *fill after running* |
| ROUGE-L Recall | *fill after running* |
| Classification Accuracy | *fill after running* |

After fixing any failures found in V1, re-run the notebook with the improved prompt to populate V2 columns and compare.